In [ ]:
!git clone https://github.com/LaboratorioSperimentale/Semantica2025.git

Cloning into 'Semantica2025'...
remote: Enumerating objects: 4326, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (36/36), done.
Receiving objects: 100% (4326/4326), 44.76 MiB | 9.88 MiB/s, done.
remote: Total 4326 (delta 19), reused 43 (delta 14), pack-reused 4276 (from 2)
Resolving deltas: 100% (3789/3789), done.


## Vettori statici nel probing BERT per l’identificazione delle costruzioni NPN



In questo notebook analizziamo il ruolo dei **vettori statici** in un *probing task*
per l’identificazione delle costruzioni **NPN** (*Noun–Preposition–Noun*).

I vettori statici vengono usati come **baseline**, da confrontare con le
rappresentazioni contestuali di **BERT**.


## Che cosa sono i vettori statici

Un vettore statico è una rappresentazione numerica associata unicamente a una parola o a un lemma.

Caratteristica fondamentale:

 **diverse occorrenze del lemma = unico vettore**

Il vettore:
- non cambia in base al contesto
- non dipende dalla frase in cui la parola compare


## Uso dei vettori statici nel probing NPN

Nel probing per l’identificazione delle NPN:

- l’input del classificatore è una rappresentazione vettoriale
- questa rappresentazione è costruita **a partire dai vettori dei lemma**
- nessuna informazione contestuale viene fornita al modello

Il classificatore deve decidere se una sequenza
realizza o meno una **costruzione NPN** basandosi solo sulla semnantica lessicale.


## Perché usare i vettori statici nel probing

I vettori statici sono fondamentali come **baseline interpretativa**.

Permettono di stimare:
- quanto l’identificazione delle NPN dipenda dal solo lessico
- quanto miglioramento derivi dall’informazione contestuale di BERT

## VETTORI STATICI

In [ ]:
import random

random.seed(42)

vec_lun = 128

embeddings_file = "/content/Semantica2025/data/static_vectors.txt"
train_file = "/content/Semantica2025/data/ex1_simple_train_1.csv"

## FUNZIONI

In [ ]:
def get_embedding(lemma, vettori, oov_vectors, vector_dim):
  """
  Restituisce embedding del `lemma' se presente in vettori, altrimenti un vettore
  random di `vector_dim' dimensioni. Contestualmente, lo aggiunge al
  dizionario `oov_vectors'
  """
  if lemma in vettori:
    #se il lemma è in vettori, ritorna il vettore corrispondente convertito in numpy.array

    return np.array(vettori[lemma])

  if lemma not in oov_vectors:
    #se il lemma non è in vettori e nemmeno è già stato registrato in oov_vectors, allora bisogna crearne uno nuovo
    oov_vectors[lemma] = np.random.rand(vec_lun)

  return oov_vectors[lemma]


def load_vectors(vector_file, lemmas_list):
  """
  Restituisce il dizionario vettori contenente solo i lemma presenti in
  lemmas_list e i loro vettori (liste di float).
  """
  vettori = {}
  with open(vector_file) as fin:
    for line in fin:
      colonne = line.strip().split("\t")
      lemma = colonne[0]

      if lemma in lemmas_list:
        vettori[lemma] = []
        lunghezza_vettori = len(colonne) - 1
        for valore in colonne[1:len(colonne)]:
          vettori[lemma].append(float(valore))

  return vettori



In [ ]:
# Per i vettori ITWAC

import csv, numpy as np
from sklearn.model_selection import GroupKFold

dataset = []
with open(train_file, encoding="utf-8") as f:
  train_csv = csv.DictReader(f, delimiter=";")
  for item in train_csv:
    lemma = item["noun"].strip()

    dataset.append([item["construction"].strip(), lemma, item["costr"].strip()])

    #delimitare dal data set iniziale i valori fondamentali
    #etichetta di costruzione (yes"/"no")



lemma_numbers = {}
#lemma_numbers = {}: dizionario che mappa ogni lemma incontrato a un intero univoco (usato per i gruppi)
X, y, lemma_groups = [], [], []

for construction_label, lemma, costr in dataset:

  if lemma not in lemma_numbers:
    lemma_numbers[lemma] = len(lemma_numbers)

  X.append(costr)

  if construction_label=="yes":
    y.append(1)
  else:
    y.append(0)

  lemma_groups.append(lemma_numbers[lemma])


vettori = load_vectors(embeddings_file, list(lemma_numbers.keys()))
oov_vectors = {}

folds = []
gkf = GroupKFold(n_splits=5, shuffle=True)

for train_idx, test_idx in gkf.split(X, y, groups=lemma_groups):

  #print("TRAIN", train_idx)
  #print("TEST", test_idx)

  # X_train = [], y_train = []: inizializza liste vuote per memorizzare
  # i vettori e le etichette del train set del fold corrente
  X_train = []
  y_train = []
  for id in train_idx:
    lemma = dataset[id][1]
    X_train.append(get_embedding(lemma, vettori, oov_vectors, vec_lun))
    y_train.append(y[id])

  X_test = []
  y_test = []
  for id in test_idx:
    lemma = dataset[id][1]
    X_test.append(get_embedding(lemma, vettori, oov_vectors, vec_lun))
    y_test.append(y[id])

  folds.append({
      "train_X": np.array(X_train),
      "train_y": np.array(y_train),
      "test_X":  np.array(X_test),
      "test_y":  np.array(y_test)
  })


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

accuracies = []

for i, fold in enumerate(folds):
    print(f"\n=== Fold {i+1} ===")

    train_X = fold["train_X"]
    train_y = fold["train_y"]
    test_X  = fold["test_X"]
    test_y  = fold["test_y"]

    clf = LogisticRegression(max_iter=10000, solver='liblinear')
    clf.fit(train_X, train_y)

    accuracy = clf.score(test_X, test_y)
    accuracies.append(accuracy)

    print(f"Accuracy fold {i+1}: {accuracy:.4f}")

# media delle accuracy
mean_accuracy_baseline = np.mean(accuracies)

print("\n=== Accuracy media sui 5 fold ===")
print(f"Accuracy media: {mean_accuracy_baseline:.4f}")



=== Fold 1 ===
Accuracy fold 1: 0.6134

=== Fold 2 ===
Accuracy fold 2: 0.7979

=== Fold 3 ===
Accuracy fold 3: 0.9018

=== Fold 4 ===
Accuracy fold 4: 0.6512

=== Fold 5 ===
Accuracy fold 5: 0.8842

=== Accuracy media sui 5 fold ===
Accuracy media: 0.7697


## BERT

## SET UP MODELLO E TOKENIZER

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from transformers import AutoTokenizer, AutoModelForMaskedLM


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")
# se è disponibile una GPU (cuda) viene usata, altrimenti la CPU
# device è usato per spostare modello e tensori.

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-italian-cased")
# viene caricato il tokenizer e il modello BERT per Masked Language Modeling

model = AutoModelForMaskedLM.from_pretrained(
    "dbmdz/bert-base-italian-cased",
    output_hidden_states=True
    # output_hidden_states=True fa sì che model(...) ritorni anche
    # tutti gli hidden states di tutti i layer
).to(device)
model.eval()
# imposta il modello in modalità inference (disabilita dropout ecc.)

tokenizer.save_pretrained("data/tokenizer")
# salva il tokenizer localmente in data/tokenizer
# necessario perchè poi sarà richiamato

Usando device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

## FUNZIONI

In [ ]:
def substitute_unknown(context_pre, costr, context_post):

  '''
  GENERA DUE VERSIONI DIVERSE DELLA FRASE

  UNA SOSTIUISCE LA PREPOSIZIONE CON EMBEDDING SPECIALE [UNK]
  UNA MANTIENE L'OCCORRENZA ORIGINALE

  '''

  lemma1, prep, lemma2 = costr.strip().split(" ")
  sentence = f"{context_pre} {lemma1} [UNK] {lemma2} {context_post}"
  sentence_orig = f"{context_pre} {costr} {context_post}"
  #calcola una posizione in termini di caratteri
  #WARNING = La tokenizzazione di BERT è a subword: questo può portare a errori quando si cerca di mappare caratteri
  posizione_preposizione = len(lemma1) + len([x for x in context_pre if not x == " "])

  return {
      "sentence": sentence,
      "sentence_orig": sentence_orig,
      "posizione_preposizione": posizione_preposizione
  }

  #restituisce un oggetto dizionario che ha vome chiave le variabili indicate
  #come oggetti le frasi (modificata e originale) e la posizione preposizione

In [ ]:
def get_BERT_embedding(sentence, sentence_orig, posizione_preposizione, tokenizer, model, device):
    """
Questa funzione esegue la tokenizzazione e l'inferenza su BERT e estrae, per ogni layer, gli embeddings di:
il token [UNK] nella frase con [UNK] (sentence) — embeddings_UNK
il token [CLS] nella stessa frase — embeddings_CLS
il token della preposizione nella frase originale (sentence_orig) — embeddings_PREP
    """
    # tokenizzare le frasi in output dalla funzione substitute_unknown
    inputs = tokenizer(sentence, return_tensors="pt").to(device)
    inputs_orig = tokenizer(sentence_orig, return_tensors="pt").to(device)

    #CERCHIAMO LA POSIZIONE DELLA PREPOSIZIONE PER ESTRARRE IL VETTORE DI QUEL TOKEN

    tokens = tokenizer.tokenize(sentence_orig)
    #da ID numerici a lista di stringhe
    tot_car = 0
    #inizializza un contatore che dovrebbe accumulare il numero di caratteri
    i = 0
    while i < len(tokens) and tot_car < posizione_preposizione:
        curr_chars = len([x for x in tokens[i] if x != "#"])
        tot_car += curr_chars
        i += 1
    #trovare l'indice del token della preposizione sommando i caratteri token per token fino a raggiungere posizione_preposizione
    prep_index = i + 1
    #il primo token è sempre CLS

    embeddings_UNK, embeddings_CLS, embeddings_PREP = [], [], []

    with torch.no_grad():
      #torch.no_grad() disabilita il calcolo dei gradienti (inferenza).
        outputs = model(**inputs)
        output_orig = model(**inputs_orig)


        target_id_UNK = inputs["input_ids"][0].tolist().index(tokenizer.unk_token_id)
        #CERCHIAMO LA POSIZIONE DI [UNK]
        #LA POSIZIONE DI [CLS] è SEMPRE [0]

        for layer in range(1, 13):

            hidden_UNK = outputs.hidden_states[layer][0, target_id_UNK, :].cpu().numpy()
            hidden_CLS = outputs.hidden_states[layer][0, 0, :].cpu().numpy()

            #.cpu().numpy() sposta i tensori su CPU e li converte in numpy array
            hidden_PREP = output_orig.hidden_states[layer][0, prep_index, :].cpu().numpy()
            embeddings_UNK.append(hidden_UNK)
            embeddings_CLS.append(hidden_CLS)
            embeddings_PREP.append(hidden_PREP)

    return {
        "embeddings_UNK": embeddings_UNK,
        "embeddings_CLS": embeddings_CLS,
        "embeddings_PREP": embeddings_PREP
    }

## CARICAMENTO DATI

In [ ]:
import pandas as pd

train_file = "/content/Semantica2025/data/ex1_simple_train_1.csv"
df = pd.read_csv(train_file, sep=";")
print(f"File letto: {len(df)} righe")

File letto: 506 righe


## INIZIALIZZAZIONE DELLE LISTE

In [ ]:
X, y, groups, lemma_numbers = [], [], [], {}
bert_embeddings = []

#X: lista che conterrà le rappresentazioni dei contesti di occorrenza
#y: lista delle etichette binarie (1 se l'occorrenza contiene una costruzione NPN, 0 altrimenti)
#groups: lista di interi che indicano il gruppo a cui appartiene ogni esempio
#lemma_numbers: dizionario che assegna a ciascun lemma un numero unico (per il grouping con GroupKFold)
#bert_embeddings: lista che conterrà gli embedding BERT calcolati per ogni esempio

##

In [ ]:
for idx,(_, row ) in enumerate(df.iterrows()):
    row = row.to_dict()
    lemma = row["noun"].strip()
    if lemma not in lemma_numbers:
        lemma_numbers[lemma] = len(lemma_numbers)


    #RAPPRESENTAZIONE DELL'OCCORRENZA

    entry = substitute_unknown(row["context_pre"], row["costr"], row["context_post"])
    X.append(entry)

    #ETICHETTA BINARIA
    y.append(1 if row["construction"].strip() == "yes" else 0)
    groups.append(lemma_numbers[lemma])
    #groups diventa quindi una lista di interi dove tutte le occorrenze dello stesso lemma hanno lo stesso numero


    #EMBEDDING BERT
    emb = get_BERT_embedding(
        entry["sentence"],
        entry["sentence_orig"],
        entry["posizione_preposizione"],
        tokenizer,
        model,
        device
    )
    bert_embeddings.append(emb)

    if (idx+1) % 50 == 0 or (idx+1) == len(df):
        print(f"Righe processate: {idx+1}/{len(df)}")


Righe processate: 50/506
Righe processate: 100/506
Righe processate: 150/506
Righe processate: 200/506


## CREAZIONE FOLD

In [ ]:
y = np.array(y)

folds = []
gkf = GroupKFold(n_splits=5)

for fold_id, (train_idx, test_idx) in enumerate(gkf.split(bert_embeddings, y, groups), 1):
    print(f"\n=== Creazione Fold {fold_id} ===")

    X_train = [bert_embeddings[i] for i in train_idx]
    X_test = [bert_embeddings[i] for i in test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    folds.append({
        "train_X_UNK":  np.array([x["embeddings_UNK"] for x in X_train]),
        "train_X_PREP": np.array([x["embeddings_PREP"] for x in X_train]),
        "train_X_CLS":  np.array([x["embeddings_CLS"] for x in X_train]),
        "test_X_UNK":   np.array([x["embeddings_UNK"] for x in X_test]),
        "test_X_PREP":  np.array([x["embeddings_PREP"] for x in X_test]),
        "test_X_CLS":   np.array([x["embeddings_CLS"] for x in X_test]),
        "train_y": y_train,
        "test_y": y_test
    })

print(f"\nPreparati {len(folds)} fold.")
print(f"Esempio shape UNK del primo fold: {folds[0]['train_X_UNK'].shape}")


=== Creazione Fold 1 ===

=== Creazione Fold 2 ===

=== Creazione Fold 3 ===

=== Creazione Fold 4 ===

=== Creazione Fold 5 ===

Preparati 5 fold.
Esempio shape UNK del primo fold: (404, 12, 768)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

embedding_types = ["UNK", "PREP", "CLS"]
num_layers = 12  # numero di layer in BERT

# Dizionario per memorizzare accuracies
layer_accuracies = {etype: {layer: [] for layer in range(num_layers)} for etype in embedding_types}

# Itera sui fold
for i, fold in enumerate(folds):
    print(f"\n=== Fold {i+1} ===")
    train_y = fold["train_y"]
    test_y  = fold["test_y"]

    for etype in embedding_types:
        for layer in range(num_layers):
            # Estrai le feature del layer corrispondente
            train_X_layer = np.array([emb[layer] for emb in fold[f"train_X_{etype}"]])
            test_X_layer  = np.array([emb[layer] for emb in fold[f"test_X_{etype}"]])

            clf = LogisticRegression(max_iter=10000, solver='liblinear')
            clf.fit(train_X_layer, train_y)
            preds = clf.predict(test_X_layer)
            acc = accuracy_score(test_y, preds)
            layer_accuracies[etype][layer].append(acc)

# Calcola media accuracy per layer
mean_accuracies = {etype: [np.mean(layer_accuracies[etype][layer]) for layer in range(num_layers)]
                   for etype in embedding_types}

for el in mean_accuracies:
  print("====\n" +
        el + "====\n"
        + str(mean_accuracies[el])
  )




=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===
====
UNK====
[np.float64(0.8498349834983498), np.float64(0.8437973209085614), np.float64(0.8497767423801204), np.float64(0.8380120364977675), np.float64(0.8339351582217045), np.float64(0.891166763735197), np.float64(0.9189477771306542), np.float64(0.9129877693651718), np.float64(0.9149485536788973), np.float64(0.9170064065230052), np.float64(0.9348670161133761), np.float64(0.9290040768782759)]
====
PREP====
[np.float64(0.8359930110658125), np.float64(0.8062317996505532), np.float64(0.8398369248689574), np.float64(0.8556979227334498), np.float64(0.8496990875558144), np.float64(0.9247913026596777), np.float64(0.9545525140749369), np.float64(0.9564744709765094), np.float64(0.9644535041739468), np.float64(0.9565909532129684), np.float64(0.9546689963113959), np.float64(0.942787808192584)]
====
CLS====
[np.float64(0.6880993981751116), np.float64(0.6919433119782566), np.float64(0.7136478353717723), np.float64(0.69

In [ ]:
# Stampa e grafico
plt.figure(figsize=(10, 5))
for etype in embedding_types:
    plt.plot(range(1, num_layers+1), mean_accuracies[etype], marker='o', label=etype)

plt.axhline(
    y=baseline_accuracy,
    linestyle="--",
    label="Static embeddings (baseline)"
)

plt.xlabel("Layer BERT")
plt.ylabel("Accuracy media sui fold")
plt.ylim(0.3, 1.0)
plt.title("Performance Logistic Regression per layer BERT")
plt.xticks(range(1, num_layers+1))
plt.grid(True)
plt.legend()
plt.show()


NameError: name 'plt' is not defined